# Notebook 37 — Agent Evaluation, Sandboxing, and Trajectory Observability

    ## Learning objectives

    - Evaluate decisions, trajectories, effects, and resource use rather than final text alone
- Build deterministic simulated tools, replay tests, fault injection, and approval assertions
- Threat-model browser, coding, computer-use, and long-running agents with enforceable sandboxes

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if IN_COLAB and not token:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
        if token:
            break
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub variable. The course also sets its
# descriptive alias because some lesson code uses HUGGINGFACE_TOKEN explicitly.
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("Add an HF_TOKEN secret in Colab, enable notebook access, then rerun this cell.")


## 37.1 The trajectory is the product behavior

Two agents can return the same answer while one leaked data, called unnecessary tools, ignored a denial, or spent
ten times the budget. Evaluation units include task, decision step, tool call, observation, state transition,
external effect, final answer, and complete run. Preserve structured traces with model/template/tool/policy versions
and authenticated effect receipts.

Define success and forbidden outcomes before collecting examples. Final correctness, groundedness, tool selection,
argument validity, call precision/recall, recovery, step count, latency, tokens, cost, side effects, and human
interventions need separate measures. Never average a catastrophic unauthorized action into a good overall score.


In [ ]:
trace = [{"kind":"model_action","tool":"lookup","args":{"id":"A"}},
         {"kind":"tool_result","tool":"lookup","status":"ok","value":17},
         {"kind":"model_action","tool":"calculator","args":{"expression":"17*2"}},
         {"kind":"tool_result","tool":"calculator","status":"ok","value":34},
         {"kind":"final","answer":"34"}]
expected_tools = {"lookup","calculator"}
actual_tools = {event["tool"] for event in trace if event["kind"]=="model_action"}
print({"tool_recall":len(actual_tools&expected_tools)/len(expected_tools),
       "unnecessary_tools":len(actual_tools-expected_tools), "steps":len(trace)})


## 37.2 Deterministic environments and replay

Replace live tools with versioned fakes in CI. A scenario defines initial state, observations, permitted actions,
injected failures, hidden assertions, and terminal conditions. Replay captured model actions into the same fake to
test policy and executor changes; replay captured tool results into the planner to test model/prompt changes.
Separating these halves makes regressions diagnosable.

Golden trajectories are not the only valid path, so assert invariants and acceptable effects rather than exact
prose. Metamorphic tests rename identifiers, reorder irrelevant context, perturb formatting, and vary transient
failures while preserving the required outcome. Seeded sampling improves comparison but does not promise identical
outputs across runtimes.


In [ ]:
class FakeLedger:
    def __init__(self): self.calls=[]; self.fail_once=True
    def lookup(self, item_id):
        self.calls.append(("lookup",item_id))
        if self.fail_once: self.fail_once=False; raise TimeoutError("injected")
        return {"id":item_id,"value":17}
ledger=FakeLedger()
for attempt in range(2):
    try: print(ledger.lookup("A")); break
    except TimeoutError: print("bounded retry", attempt+1)
assert ledger.calls == [("lookup","A"),("lookup","A")]


## 37.3 Fault and adversarial matrix

Inject malformed model JSON, unknown tools, schema violations, oversized observations, tool timeouts, rate limits,
stale state, duplicate delivery, partial streams, worker crashes, permission revocation, cancellation, poisoned
memory, malicious retrieved text, and conflicting agents. Expected behavior may be repair, bounded retry, fallback,
approval, abstention, or safe failure. It is never an unbounded loop.

Red-team indirect prompt injection from webpages, files, email, issues, documents, MCP resources, and tool errors.
Ensure untrusted content cannot change system policy, reveal secrets, expand tools, or authorize effects. Maintain
exploit fixtures in regression suites after remediation.


In [ ]:
cases = [{"fault":"unknown_tool","expected":"reject"}, {"fault":"timeout","expected":"bounded_retry"},
         {"fault":"permission_revoked","expected":"deny"}, {"fault":"injection_in_result","expected":"treat_as_data"},
         {"fault":"duplicate_mutation","expected":"return_idempotent_receipt"}]
print(*cases, sep="\n")


## 37.4 Browser, computer-use, and coding sandboxes

Browser/computer agents observe partial, changing state. DOM elements move, screenshots become stale, pages contain
hostile instructions, and clicks can purchase, publish, or delete. Re-observe before consequential actions, bind
approvals to current targets, constrain navigation and downloads, isolate cookies, and confirm effects through
trusted application state. Visual text is untrusted just like retrieved text.

Coding agents require disposable processes or VMs/containers with explicit filesystem mounts, no host credentials,
denied network by default, CPU/memory/process/time/output limits, dependency policy, and artifact scanning. A Python
import allowlist is not an operating-system sandbox. Preserve patches and test results; require approval before
publishing, deploying, messaging, or modifying resources outside the workspace.


In [ ]:
sandbox_policy = {"filesystem":"ephemeral + explicit read-only inputs", "network":"deny by default",
                  "secrets":"none", "cpu_seconds":30, "memory_mib":512, "processes":16,
                  "output_kib":256, "external_writes":"approval gateway"}
print(sandbox_policy)


## 37.5 Observability without surveillance

Trace model calls, validation, policy, queueing, tools, approvals, and effects with correlation IDs and durations.
Record model/template/tool/policy revisions, token counts, retries, stop reason, state version, and redacted error
classes. Metrics include success by slice, critical violation rate, tool validity, denial adherence, loop exhaustion,
latency percentiles, total tokens, and intervention rate.

Prompts, memory, screenshots, files, and tool results can contain personal data and secrets. Minimize collection,
redact before export, separate restricted payloads from operational metadata, encrypt, restrict access, define
retention/deletion, and audit trace viewers. Sampling full content “for debugging” is a data system requiring review.


In [ ]:
events = [{"run":"r1","status":"success","tokens":800,"tool_calls":2,"violations":0},
          {"run":"r2","status":"failed","tokens":1600,"tool_calls":8,"violations":0},
          {"run":"r3","status":"failed","tokens":400,"tool_calls":1,"violations":1}]
print({"success_rate":sum(e["status"]=="success" for e in events)/len(events),
       "critical_violation_rate":sum(e["violations"]>0 for e in events)/len(events),
       "tokens_per_success":sum(e["tokens"] for e in events)/max(1,sum(e["status"]=="success" for e in events))})


## 37.6 Release evaluation

Build task suites from representative workflows, rare consequential actions, production failures, and adversarial
scenarios. Split development from frozen release gates. Compare prompts, models, frameworks, and multi-agent designs
under equal permissions and realistic token/time budgets. Use paired runs and uncertainty; inspect slice regressions.

Before promotion, require zero known critical authorization/effect violations, bounded behavior under every injected
fault, tested cancellation and resume, sandbox escape review, quality and latency thresholds, and a rollback drill.
Canary with limited scopes and monitor effect-level outcomes. Agent evaluation is continuous because tools, websites,
schemas, permissions, and models change independently.


## 37.7 Outcome, trajectory, and effect scoring

Final-answer correctness misses whether an agent used the right tool, supplied valid arguments, respected permissions, duplicated effects, recovered safely, or wasted resources. Score outcomes, intermediate decisions, environment state, critical violations, steps, tokens, latency, and human interventions. Prefer deterministic environment checks and tool receipts; use model judges only for clearly defined semantic dimensions. Preserve full redacted traces for paired failure analysis.


In [ ]:
episode={"task_success":1,"tool_precision":2/3,"argument_validity":1,"duplicate_effects":0,"critical_violations":0,"steps":5,"tokens":620}; print(episode)


## 37.8 Sandboxing is defense in depth

A sandbox controls processes, filesystem, network, resources, time, and credentials; a prompt saying not to escape is not a sandbox. Start from no network and a minimal read-only filesystem, add explicit mounts and destinations, cap CPU/memory/processes/output, and destroy the environment after use. Keep authorization outside the sandbox and assume generated code is hostile. Test symlinks, path traversal, fork bombs, environment-variable access, encoded payloads, dependency installation, and exfiltration channels.


In [ ]:
policy={"network":"deny","filesystem":{"read":["/workspace/input"],"write":["/workspace/output"]},"cpu_seconds":5,"memory_mb":256,"processes":8,"secrets":[]}; print(policy); assert policy["network"]=="deny" and not policy["secrets"]


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [AgentBench](https://arxiv.org/abs/2308.03688)
- [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)


## Exercises

    1. Create a deterministic scenario runner with ten fault injections and invariant assertions.
2. Compare single-agent and multi-agent trajectories under equal budgets.
3. Threat-model a coding agent sandbox and demonstrate why language-level restrictions are insufficient.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
